# Ascon v2 — Caminho B (CNN1D) — smoke test

Antes de rodar:
1. **Settings (painel direito) → Accelerator → GPU T4 x2**
2. **Add Input → Datasets** → anexe o Dataset do Kaggle com
   `keyholdout_5class_v2.parquet` (11,8GB) e `v2_folds.json` (30KB).
3. Ajuste `DATA_DIR` na célula de symlinks abaixo pro slug real do seu Dataset
   (aparece em `/kaggle/input/<slug>` depois de anexado).

Gate obrigatório antes de qualquer CV — mede tempo/época, VRAM e parâmetros,
e extrapola o custo da CV completa (ver `docs/plano_experimento_v2/07_runbook_execucao.md`).

In [ ]:
import subprocess, sys, os, glob, json

print("=== VERIFICAÇÃO DE AMBIENTE ===")

import torch
print(f"PyTorch : {torch.__version__}")
print(f"CUDA    : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU     : {torch.cuda.get_device_name(0)}")
    print(f"VRAM    : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    raise RuntimeError(
        "GPU não detectada!\n"
        "Ative em: Settings (painel direito) → Accelerator → GPU T4 x2"
    )

import psutil
print(f"RAM     : {psutil.virtual_memory().total / 1e9:.1f} GB")
print(f"Python  : {sys.version.split()[0]}")

In [ ]:
# ═══════════════════════════════════════════════════════════
# Dependências
# ═══════════════════════════════════════════════════════════
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "pyarrow", "scikit-learn", "xgboost",
], check=True)
print("Dependências instaladas.")

In [ ]:
# ═══════════════════════════════════════════════════════════
# Clonar repositório
# ═══════════════════════════════════════════════════════════
REPO_DIR = "/kaggle/working/ascon"

if not os.path.exists(REPO_DIR):
    result = subprocess.run(
        ["git", "clone", "https://github.com/K1nginthen0rth/ascon.git"],
        capture_output=True, text=True,
        cwd="/kaggle/working"
    )
    print(result.stderr.strip())
    if result.returncode != 0:
        raise RuntimeError("Falha ao clonar repositório.")
    print("Repositório clonado.")
else:
    print("Repositório já existe — atualizando...")
    result = subprocess.run(["git", "pull"], capture_output=True, text=True, cwd=REPO_DIR)
    print(result.stdout.strip())
    print(result.stderr.strip())

In [ ]:
# ═══════════════════════════════════════════════════════════
# Symlinks do dataset v2 — AJUSTAR DATA_DIR pro slug do seu Kaggle Dataset
# ═══════════════════════════════════════════════════════════
DATA_DIR = "/kaggle/input/<slug-do-seu-kaggle-dataset-v2>"  # <-- AJUSTAR

os.makedirs(f"{REPO_DIR}/data/processed", exist_ok=True)
links = {
    f"{REPO_DIR}/data/processed/keyholdout_5class_v2.parquet": f"{DATA_DIR}/keyholdout_5class_v2.parquet",
    f"{REPO_DIR}/data/processed/v2_folds.json":                f"{DATA_DIR}/v2_folds.json",
}
for link, target in links.items():
    if os.path.islink(link) or os.path.exists(link):
        os.remove(link)
    os.symlink(target, link)
    print(f"Symlink criado: {os.path.basename(link)} -> existe={os.path.exists(link)}")

In [ ]:
# ═══════════════════════════════════════════════════════════
# Smoke test Caminho B (CNN1D)
# ═══════════════════════════════════════════════════════════
os.chdir(REPO_DIR)
result = subprocess.run([
    sys.executable, "scripts/run_v2_caminhos_bce.py",
    "--path", "B", "--mode", "smoke", "--branch", "controlado",
], cwd=REPO_DIR)
print(f"\nreturncode={result.returncode}")
if result.returncode != 0:
    raise RuntimeError("Smoke test do Caminho B falhou — ver output acima.")

In [ ]:
# ═══════════════════════════════════════════════════════════
# Métricas + matriz de confusão (F1, accuracy, AUC-ROC, precision/recall por classe)
# ═══════════════════════════════════════════════════════════
from IPython.display import Image, display

out_dir = f"{REPO_DIR}/reports/v2/caminho_b/controlado"

for jf in sorted(glob.glob(f"{out_dir}/*_metrics.jsonl")):
    print(f"\n=== {os.path.basename(jf)} ===")
    with open(jf, encoding="utf-8") as fh:
        for line in fh:
            rec = json.loads(line)
            print(json.dumps(rec, indent=2, ensure_ascii=False))

for png in sorted(glob.glob(f"{out_dir}/confusion_matrices/*smoke*.png")):
    print(png)
    display(Image(filename=png))